# Phase 4 · CJPE · probability-averaging ensemble
Reads `dev_probs.json` / `test_probs.json` exported by each run notebook (cell 6), averages the probabilities, picks the threshold on dev, and writes one `submission.zip`. No GPU, no model loading. Edit `RUN_DIRS`.

In [ ]:
from pathlib import Path
import json, zipfile, numpy as np
COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course'); RUNS = COURSE_ROOT/'runs/phase4_cjpe'
from google.colab import drive; drive.mount('/content/drive')
RUN_DIRS = [RUNS/'MHA-prenorm/full/mha-full-001', RUNS/'GQA-prenorm/lorma/gqa-lorma-001', RUNS/'MLA-prenorm/lora/mla-lora-001', RUNS/'GQA-prenorm/kd/gqa-kd-001']
WEIGHTS = None            # e.g. [2,1,1,1]
OUT = RUNS/'ensemble/ens-001'; OUT.mkdir(parents=True, exist_ok=True)
PHASE_DIR = COURSE_ROOT/'Phase_4_CJPE'
dev = {d['id']: int(d['label']) for d in json.loads((PHASE_DIR/'Train/dev.json').read_text())} if isinstance(json.loads((PHASE_DIR/'Train/dev.json').read_text()), list) else None
if dev is None:
    raw = json.loads((PHASE_DIR/'Train/dev.json').read_text()); dev = {str(k): int(v.get('label', v.get('labels'))) for k, v in raw.items()}
test_ids = [str(d['id']) for d in json.loads((PHASE_DIR/'Test/test.json').read_text())] if isinstance(json.loads((PHASE_DIR/'Test/test.json').read_text()), list) else list(json.loads((PHASE_DIR/'Test/test.json').read_text()))
P = {s: [json.loads((r/f'{s}_probs.json').read_text()) for r in RUN_DIRS] for s in ('dev', 'test')}
dev_ids = list(P['dev'][0]); assert all(list(p) == dev_ids for p in P['dev']) and set(dev_ids) == set(dev), 'dev ids differ between runs'
assert all(set(p) == set(test_ids) for p in P['test']), 'test ids differ between runs'
y = np.array([dev[i] for i in dev_ids])
def macro_f1(y, p):
    f = []
    for c in (0, 1):
        tp = ((p == c) & (y == c)).sum(); fp = ((p == c) & (y != c)).sum(); fn = ((p != c) & (y == c)).sum()
        pr = tp/(tp+fp) if tp+fp else 0; rc = tp/(tp+fn) if tp+fn else 0; f.append(2*pr*rc/(pr+rc) if pr+rc else 0)
    return float(np.mean(f))
w = np.array(WEIGHTS or [1.0]*len(RUN_DIRS)); w = w/w.sum()
def blend(split, ids): return sum(wi*np.array([p[i] for i in ids]) for wi, p in zip(w, P[split]))
for r, p in zip(RUN_DIRS, P['dev']): print(f'{r.name:24s} dev macro-F1 @0.5 =', round(macro_f1(y, (np.array([p[i] for i in dev_ids]) >= .5).astype(int)), 4))
pd_ = blend('dev', dev_ids)
grid = np.linspace(0.05, 0.95, 91); sc = [macro_f1(y, (pd_ >= t).astype(int)) for t in grid]; thr = float(grid[int(np.argmax(sc))])
print('ENSEMBLE dev macro-F1 @0.5 =', round(macro_f1(y, (pd_ >= .5).astype(int)), 4), '| tuned', round(max(sc), 4), f'@ {thr:.2f}')
pt = blend('test', test_ids)
for tag, t in (('submission', 0.5), ('submission_tuned', thr)):
    d = OUT/tag; d.mkdir(exist_ok=True); preds = [{'id': i, 'pred': int(p >= t)} for i, p in zip(test_ids, pt)]
    (d/'prediction.json').write_text(json.dumps(preds))
    with zipfile.ZipFile(d/'submission.zip', 'w', zipfile.ZIP_DEFLATED) as z: z.write(d/'prediction.json', arcname='prediction.json')
    print(tag, 'threshold', round(t, 2), '| share predicted 1:', round(float(np.mean([p['pred'] for p in preds])), 3), '->', d/'submission.zip')